In [1]:
!pip -q install transformers torch tqdm

In [2]:
import os
import re
import gc
import json
import math
import hashlib
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModel
from google.colab import files

In [3]:
os.makedirs("data/07_finbert_updated", exist_ok=True)
os.makedirs("data/03_mda_text_updated", exist_ok=True)
os.makedirs("data/98_cache_updated", exist_ok=True)

In [4]:
dataset_v2 = pd.read_csv("dataset_v2_price_plus_dictionary_updated.csv")

print("dataset_v2 shape:", dataset_v2.shape)
print(dataset_v2.columns.tolist())
dataset_v2.head()

dataset_v2 shape: (2247, 23)
['ticker', 'cik', 'filing_date', 'filing_type', 'accession_number', 'year', 'quarter', 'cik_nolead', 'acc_nodash', 'mda_path', 'text_length_words', 'mda_status', 'primary_doc', 'past_return_10d', 'past_realized_vol_5d', 'past_realized_vol_10d', 'future_realized_vol_10d', 'abs_past_return_10d', 'lm_negative', 'lm_positive', 'lm_uncertainty', 'lm_total_words', 'lm_net_sentiment']


,ticker,cik,filing_date,filing_type,accession_number,year,quarter,cik_nolead,acc_nodash,mda_path,...,past_return_10d,past_realized_vol_5d,past_realized_vol_10d,future_realized_vol_10d,abs_past_return_10d,lm_negative,lm_positive,lm_uncertainty,lm_total_words,lm_net_sentiment
0,AAPL,320193,2019-01-30,10-Q,0000320193-19-000010,2019,1,320193,32019319000010,AAPL_20190130_10-Q_000032019319000010.txt,...,0.031200,0.018383,0.016375,0.012985,0.031200,0.040979,0.005424,0.034711,8297,-0.035555
1,AAPL,320193,2019-05-01,10-Q,0000320193-19-000066,2019,2,320193,32019319000066,AAPL_20190501_10-Q_000032019319000066.txt,...,0.007228,0.008079,0.010983,0.022134,0.007228,0.040265,0.005329,0.033989,8444,-0.034936
2,AAPL,320193,2019-07-31,10-Q,0000320193-19-000076,2019,3,320193,32019319000076,AAPL_20190731_10-Q_000032019319000076.txt,...,0.020931,0.006725,0.011068,0.028255,0.020931,0.040797,0.005400,0.034677,8334,-0.035397
3,AAPL,320193,2020-01-29,10-Q,0000320193-20-000010,2020,1,320193,32019320000010,AAPL_20200129_10-Q_000032019320000010.txt,...,0.002303,0.020648,0.015755,0.021234,0.002303,0.005647,0.001694,0.010164,3542,-0.003953
4,AAPL,320193,2020-05-01,10-Q,0000320193-20-000052,2020,2,320193,32019320000052,AAPL_20200501_10-Q_000032019320000052.txt,...,0.024802,0.020703,0.023482,0.012307,0.024802,0.008942,0.002835,0.009378,4585,-0.006107


In [5]:
TEXT_ZIP = "mda_text_files_updated.zip"
TEXT_DIR = Path("data/03_mda_text_updated")

with zipfile.ZipFile(TEXT_ZIP, "r") as z:
    z.extractall(TEXT_DIR)

print("Extracted text files:", len(list(TEXT_DIR.glob("*.txt"))))
print([x.name for x in list(TEXT_DIR.glob("*.txt"))[:10]])

Extracted text files: 2265
['NKE_20191004_10-Q_000032018719000071.txt', 'EMR_20201116_10-K_000003260420000041.txt', 'ICE_20200206_10-K_000157194920000003.txt', 'MAR_20210510_10-Q_000162828021009633.txt', 'ECL_20211029_10-Q_000155837021013798.txt', 'CAT_20190506_10-Q_000001823019000153.txt', 'BMY_20231026_10-Q_000001427223000181.txt', 'XOM_20231031_10-Q_000003408823000056.txt', 'ORCL_20241210_10-Q_000095017024134973.txt', 'ISRG_2019-02-04_10-K_0001035267-19-000012_ULTRA.txt']


In [6]:
dataset_v2["mda_filename"] = dataset_v2["mda_path"].astype(str).apply(lambda x: os.path.basename(x))
dataset_v2["file_exists"] = dataset_v2["mda_filename"].apply(lambda x: (TEXT_DIR / x).exists())

print(dataset_v2["file_exists"].value_counts(dropna=False))
print("Missing files:", (~dataset_v2["file_exists"]).sum())

file_exists
True    2247
Name: count, dtype: int64
Missing files: 0


In [7]:
missing_rows = dataset_v2.loc[~dataset_v2["file_exists"]].copy()
missing_rows.to_csv("data/07_finbert_updated/step7_missing_text_rows_updated.csv", index=False)

print("Missing rows saved:", len(missing_rows))
missing_rows.head()

Missing rows saved: 0


,ticker,cik,filing_date,filing_type,accession_number,year,quarter,cik_nolead,acc_nodash,mda_path,...,past_realized_vol_10d,future_realized_vol_10d,abs_past_return_10d,lm_negative,lm_positive,lm_uncertainty,lm_total_words,lm_net_sentiment,mda_filename,file_exists


In [8]:
dataset_v2 = dataset_v2.loc[dataset_v2["file_exists"]].copy().reset_index(drop=True)

print("Rows retained for FinBERT:", len(dataset_v2))

Rows retained for FinBERT: 2247


In [9]:
MODEL_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("Device:", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.bias              | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda


In [10]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [11]:
def chunk_text(text, chunk_size_words=200, min_chunk_words=20):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size_words):
        chunk = " ".join(words[i:i + chunk_size_words])
        if len(chunk.split()) >= min_chunk_words:
            chunks.append(chunk)

    return chunks

In [12]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

In [13]:
def cache_key_from_filename(fname):
    return hashlib.md5(fname.encode("utf-8")).hexdigest()

def embed_text_with_finbert(text, max_chunks=20):
    text = clean_text(text)
    chunks = chunk_text(text, chunk_size_words=200, min_chunk_words=20)

    if len(chunks) == 0:
        return None

    chunks = chunks[:max_chunks]
    chunk_embeddings = []

    with torch.no_grad():
        for chunk in chunks:
            inputs = tokenizer(
                chunk,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=512
            )

            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)

            pooled = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
            vec = pooled.squeeze(0).detach().cpu().numpy()
            chunk_embeddings.append(vec)

    return np.mean(chunk_embeddings, axis=0).astype(np.float32)

In [14]:
test_path = TEXT_DIR / dataset_v2.iloc[0]["mda_filename"]

with open(test_path, "r", encoding="utf-8", errors="ignore") as f:
    test_text = f.read()

test_vec = embed_text_with_finbert(test_text)

print(type(test_vec))
print(test_vec.shape)
print(test_vec[:10])

<class 'numpy.ndarray'>
(768,)
[-0.13617729  0.7834003  -0.10307177 -0.15058689  0.4881958   0.04343749
  0.36061403  0.68652815  0.20769544 -0.06529081]


In [15]:
test_df = dataset_v2.head(20).copy()

embeddings_test = []
statuses_test = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    fname = row["mda_filename"]
    fpath = TEXT_DIR / fname
    cache_key = cache_key_from_filename(fname)
    cache_path = f"data/98_cache_updated/{cache_key}.npy"

    try:
        if os.path.exists(cache_path):
            vec = np.load(cache_path)
            embeddings_test.append(vec)
            statuses_test.append("CACHED")
            continue

        with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

        vec = embed_text_with_finbert(text)

        if vec is None:
            embeddings_test.append(None)
            statuses_test.append("NO_EMBEDDING")
        else:
            np.save(cache_path, vec)
            embeddings_test.append(vec)
            statuses_test.append("OK")

    except Exception as e:
        embeddings_test.append(None)
        statuses_test.append(f"ERROR_{type(e).__name__}")

print(pd.Series(statuses_test).value_counts())
print("Successful test embeddings:", sum(e is not None for e in embeddings_test))

  0%|          | 0/20 [00:00<?, ?it/s]

OK    20
Name: count, dtype: int64
Successful test embeddings: 20


In [16]:
embeddings = []
statuses = []
keys = []

for i, row in tqdm(dataset_v2.iterrows(), total=len(dataset_v2)):
    fname = row["mda_filename"]
    fpath = TEXT_DIR / fname
    cache_key = cache_key_from_filename(fname)
    cache_path = f"data/98_cache_updated/{cache_key}.npy"

    try:
        if os.path.exists(cache_path):
            vec = np.load(cache_path)
            embeddings.append(vec)
            statuses.append("CACHED")
        else:
            with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()

            vec = embed_text_with_finbert(text)

            if vec is None:
                embeddings.append(None)
                statuses.append("NO_EMBEDDING")
            else:
                np.save(cache_path, vec)
                embeddings.append(vec)
                statuses.append("OK")

        keys.append({
            "row_index": i,
            "ticker": row.get("ticker", ""),
            "filing_date": row.get("filing_date", ""),
            "filing_type": row.get("filing_type", ""),
            "accession_number": row.get("accession_number", ""),
            "mda_filename": fname
        })

    except Exception as e:
        embeddings.append(None)
        statuses.append(f"ERROR_{type(e).__name__}")
        keys.append({
            "row_index": i,
            "ticker": row.get("ticker", ""),
            "filing_date": row.get("filing_date", ""),
            "filing_type": row.get("filing_type", ""),
            "accession_number": row.get("accession_number", ""),
            "mda_filename": fname
        })

    if (i + 1) % 100 == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

  0%|          | 0/2247 [00:00<?, ?it/s]

In [17]:
status_series = pd.Series(statuses, name="finbert_status")
print(status_series.value_counts(dropna=False))

finbert_status
OK        2227
CACHED      20
Name: count, dtype: int64


In [18]:
ok_mask = [isinstance(x, np.ndarray) for x in embeddings]

dataset_ok = dataset_v2.loc[ok_mask].copy().reset_index(drop=True)
keys_ok = pd.DataFrame(keys).loc[ok_mask].copy().reset_index(drop=True)
status_ok = status_series.loc[ok_mask].reset_index(drop=True)

embedding_matrix = np.vstack([x for x in embeddings if isinstance(x, np.ndarray)])

dataset_ok["finbert_status"] = status_ok

print("Final successful rows:", len(dataset_ok))
print("Embedding matrix shape:", embedding_matrix.shape)

Final successful rows: 2247
Embedding matrix shape: (2247, 768)


In [19]:
failed_rows = pd.DataFrame(keys).copy()
failed_rows["finbert_status"] = status_series.values
failed_rows = failed_rows.loc[failed_rows["finbert_status"] != "OK"]
failed_rows = failed_rows.loc[failed_rows["finbert_status"] != "CACHED"].copy()

failed_rows.to_csv("data/07_finbert_updated/finbert_failed_rows_updated.csv", index=False)

print("Failed rows saved:", len(failed_rows))
failed_rows.head()

Failed rows saved: 0


,row_index,ticker,filing_date,filing_type,accession_number,mda_filename,finbert_status


In [20]:
np.save("data/07_finbert_updated/finbert_embeddings_raw_updated.npy", embedding_matrix)
keys_ok.to_csv("data/07_finbert_updated/finbert_embedding_keys_updated.csv", index=False)

print("Saved raw embeddings and keys.")

Saved raw embeddings and keys.


In [21]:
print("dataset_ok shape:", dataset_ok.shape)
print("keys_ok shape:", keys_ok.shape)
print("embedding_matrix shape:", embedding_matrix.shape)
print("Any NaN in embeddings:", np.isnan(embedding_matrix).any())
print("Mean abs value:", np.mean(np.abs(embedding_matrix)))

dataset_ok shape: (2247, 26)
keys_ok shape: (2247, 6)
embedding_matrix shape: (2247, 768)
Any NaN in embeddings: False
Mean abs value: 0.2696049


In [22]:
dataset_ok.to_csv("data/07_finbert_updated/dataset_v2_aligned_finbert_updated.csv", index=False)
print("Saved aligned base dataset.")

Saved aligned base dataset.


In [23]:
files.download("data/07_finbert_updated/finbert_embeddings_raw_updated.npy")
files.download("data/07_finbert_updated/finbert_embedding_keys_updated.csv")
files.download("data/07_finbert_updated/finbert_failed_rows_updated.csv")
files.download("data/07_finbert_updated/dataset_v2_aligned_finbert_updated.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>